# Business analytics — Published Gold

All results use the published star schema. Windows are calendar-based and relative to the latest available sale month. Totals describe the available data; completeness requires an explicit source watermark. Type I dimensions apply current attributes to historical facts.

Set the same catalog/schema widgets used for publication and a valid `process_date`. This notebook does not load files or publish data.


In [ ]:
from pathlib import Path
import sys

# Git folders expose the package under src; deployed tasks install the wheel.
for parent in (Path.cwd(), *Path.cwd().parents):
    if (parent / "src" / "iowa_lakehouse").is_dir():
        sys.path.insert(0, str(parent / "src"))
        break


In [ ]:
from iowa_lakehouse.config import PipelineConfig
from iowa_lakehouse.storage import DeltaStore
from iowa_lakehouse import analytics

parameters = {}
for name, default in {"catalog": "workspace", "bronze_schema": "iowa_bronze", "silver_schema": "iowa_silver", "gold_schema": "iowa_gold", "volume": "iowa_files", "country": "usa", "process_date": ""}.items():
    try:
        parameters[name] = dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.text(name, default)
        parameters[name] = default
store = DeltaStore(spark, PipelineConfig(**parameters))
facts = store.read("gold", "fact_sales")
stores = store.read("gold", "dim_store")
items = store.read("gold", "dim_item")


## Monthly sales and YoY

The comparison joins the exact month one year earlier. Missing comparator months yield NULL.


In [ ]:
display(analytics.monthly_sales(facts, months=24))


## Store ranking — Three calendar months


In [ ]:
display(analytics.store_ranking(facts, stores, months=3).limit(20))


## Category ranking — Twelve calendar months


In [ ]:
display(analytics.category_ranking(facts, items, months=12).limit(20))


## Gross margin by item — Six calendar months


In [ ]:
display(analytics.item_margins(facts, items, months=6).limit(50))


## Annual sales — Confirmed complete years only

Provide `complete_through_year` only when the source confirms that year is complete. It is deliberately independent of the current date.


In [ ]:
try:
    complete_year = dbutils.widgets.get("complete_through_year").strip()
except Exception:
    dbutils.widgets.text("complete_through_year", "")
    complete_year = ""
if complete_year:
    display(analytics.annual_sales(facts, complete_through_year=int(complete_year)))
else:
    print("Set complete_through_year to a source-confirmed complete year to run this query.")


## Exploratory price–volume association

One observation per country/item/month, with at least six distinct months and nonzero price/quantity variation. This is a descriptive log-log association, without causal identification or automatic pricing recommendations.


In [ ]:
display(analytics.price_volume_association(facts, items, months=36).limit(100))
